<a href="https://colab.research.google.com/github/srushtiganesh/SIT-UofG-QC-Assignment/blob/main/BB84-Attacker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 53.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=2bb359e76e94a684a1c4f10338c5cb7645d030b1f91993b4dfa610b40beb4202
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [3]:
# ============================================================
#  BB84 Quantum Key Distribution — WITH ATTACKER (EVE)
#
#  Colab setup (run first):
#    !pip install qiskit==1.2.4
#    %pip install qiskit-aer==0.15.1
#    %pip install pylatexenc==2.10
#
#  Rules satisfied:
#    - ALL randomness comes from measuring |+> = H|0>
#    - No Python random / numpy random anywhere
#    - Eve's intercept decision is a quantum coin flip per qubit
#    - Agent blocks clearly labelled: ALICE, EVE, BOB
# ============================================================

from qiskit import QuantumCircuit, transpile
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex, plot_histogram
from qiskit.quantum_info import Operator, Statevector
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.circuit import ControlledGate
import math

simulator = BasicSimulator()


# ─────────────────────────────────────────────────────────────
#  QUANTUM RANDOM NUMBER GENERATOR
#
#  Builds an n-qubit circuit where every qubit is prepared as
#  |+> = H|0> = 1/sqrt(2)(|0> + |1>) and measured in one shot.
#  BasicSimulator caps at 24 qubits, so we batch into chunks.
#  No standard RNG is used anywhere in this file.
# ─────────────────────────────────────────────────────────────
def quantum_random_bits(n: int) -> list:
    """
    Generate n independent quantum-random bits.
    Batches into chunks of 24 to respect BasicSimulator's qubit limit.
    Each qubit is prepared as H|0> = 1/sqrt(2)(|0>+|1>).
    Returns a list of n ints, each 0 or 1.
    """
    MAX_QUBITS = 24
    bits = []
    remaining = n
    while remaining > 0:
        chunk = min(remaining, MAX_QUBITS)
        qc = QuantumCircuit(chunk, chunk)
        for i in range(chunk):
            qc.h(i)          # |0> -> 1/sqrt(2)(|0> + |1>)
            qc.measure(i, i)
        job = simulator.run(transpile(qc, simulator), shots=1)
        # Qiskit bitstring is q[chunk-1]...q[0], so reverse before reading
        bitstring = list(job.result().get_counts().keys())[0]
        bitstring = bitstring.replace(" ", "")
        bits.extend(int(b) for b in reversed(bitstring))
        remaining -= chunk
    return bits[:n]


# ─────────────────────────────────────────────────────────────
#  BASIS ENCODING / DECODING
#
#  Basis 0 = rectilinear {|0>, |1>}   labelled (+)
#  Basis 1 = diagonal    {|+>, |->}   labelled (x)
#
#  Encoding table:
#    bit=0, basis=0  ->  |0>
#    bit=1, basis=0  ->  |1>
#    bit=0, basis=1  ->  |+> = H|0>
#    bit=1, basis=1  ->  |-> = HX|0>
# ─────────────────────────────────────────────────────────────

# ╔══════════════════════════════════════════════════════════╗
# ║  AGENT: ALICE                                            ║
# ╚══════════════════════════════════════════════════════════╝
def alice_encode(bit: int, basis: int) -> QuantumCircuit:
    """
    Alice encodes one classical bit into a single-qubit state.
    The returned QuantumCircuit represents the qubit she sends
    over the quantum channel toward Bob.
    """
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)      # |0> -> |1>
    if basis == 1:
        qc.h(0)      # rotate into diagonal basis
    return qc


# ╔══════════════════════════════════════════════════════════╗
# ║  AGENT: EVE                                              ║
# ╚══════════════════════════════════════════════════════════╝
def eve_intercept(qc: QuantumCircuit, eve_basis: int) -> QuantumCircuit:
    """
    Eve measures Alice's qubit in her randomly chosen basis,
    then re-encodes her result and forwards it to Bob.

    This is the intercept-resend attack:
      - If Eve's basis matches Alice's  -> qubit forwarded correctly
      - If Eve's basis differs          -> qubit state disturbed
        Bob will get the wrong bit ~50% of the time in those cases,
        introducing an expected QBER of 0.5 * 0.5 = 25%.

    Returns a fresh QuantumCircuit encoding Eve's measured result,
    which is what Bob will actually receive.
    """
    # Eve measures the incoming qubit in her chosen basis
    qc_copy = qc.copy()
    if eve_basis == 1:
        qc_copy.h(0)      # rotate to rectilinear before measuring
    qc_copy.measure(0, 0)
    job = simulator.run(transpile(qc_copy, simulator), shots=1)
    eve_result = int(list(job.result().get_counts().keys())[0])

    # Eve re-encodes her result and sends it on to Bob
    return alice_encode(eve_result, eve_basis)


# ╔══════════════════════════════════════════════════════════╗
# ║  AGENT: BOB                                              ║
# ╚══════════════════════════════════════════════════════════╝
def bob_measure(qc: QuantumCircuit, basis: int) -> int:
    """
    Bob measures a received qubit in his chosen basis.
    Returns 0 or 1.
    """
    qc = qc.copy()
    if basis == 1:
        qc.h(0)      # undo diagonal rotation before measuring
    qc.measure(0, 0)
    job = simulator.run(transpile(qc, simulator), shots=1)
    return int(list(job.result().get_counts().keys())[0])


# ╔══════════════════════════════════════════════════════════╗
# ║  BB84 PROTOCOL — WITH EVE (intercept-resend attack)      ║
# ╚══════════════════════════════════════════════════════════╝
def run_bb84_with_attacker(n_bits: int = 100, error_threshold: float = 0.15):
    """
    BB84 key exchange with Eve performing an intercept-resend attack.

    Protocol steps
    --------------
    1. ALICE   draws random bits + bases (quantum), encodes qubits
    2. EVE     for each qubit, flips a quantum coin (quantum_random_bits(1))
               to decide whether to intercept; if yes, measures in a
               random basis and re-encodes her result toward Bob
    3. BOB     draws random bases (quantum), measures whatever he receives
    4. PUBLIC  Alice and Bob compare bases over classical channel
    5. SIFT    discard positions where bases differ
    6. CHECK   sacrifice ~20% of sifted key to estimate QBER
    7. REPORT  declare secure if QBER <= threshold, else abort

    Why Eve causes errors
    ---------------------
    When Eve intercepts, she guesses the wrong basis ~50% of the time.
    In those cases Bob's measurement is effectively random, giving him
    the wrong bit ~50% of the time even when his basis matches Alice's.
    Expected QBER from full (100%) intercept: 0.5 * 0.5 = 25%.
    """
    print("=" * 58)
    print("  BB84 — WITH EVE (intercept-resend attack)")
    print("=" * 58)

    # ----------------------------------------------------------
    # ALICE: generate random bits and bases, encode qubits
    # ----------------------------------------------------------
    print(f"\n[ALICE] Generating {n_bits} random bits and bases ...")
    alice_bits  = quantum_random_bits(n_bits)
    alice_bases = quantum_random_bits(n_bits)

    qubits_sent = [alice_encode(bit, basis)
                   for bit, basis in zip(alice_bits, alice_bases)]

    print(f"  bits  (first 10): {alice_bits[:10]}")
    print(f"  bases (first 10): {alice_bases[:10]}   (0=+, 1=x)")

    # ----------------------------------------------------------
    # EVE: sits on the quantum channel between Alice and Bob.
    # For each qubit she flips a quantum coin — quantum_random_bits(1)
    # returns [0] or [1] with equal probability, no RNG involved.
    # If the coin is 1 she intercepts; if 0 she lets it pass.
    # She also picks a random basis for each interception.
    # ----------------------------------------------------------
    print(f"\n[EVE]   Intercepting qubits (quantum coin decides per qubit) ...")
    eve_bases   = quantum_random_bits(n_bits)   # Eve's measurement bases
    eve_coins   = quantum_random_bits(n_bits)   # 1 = intercept, 0 = pass

    qubits_received_by_bob = []
    n_intercepted = 0

    for i in range(n_bits):
        if eve_coins[i] == 1:
            # Eve intercepts, measures, re-encodes, and forwards
            n_intercepted += 1
            disturbed_qubit = eve_intercept(qubits_sent[i], eve_bases[i])
            qubits_received_by_bob.append(disturbed_qubit)
        else:
            # Qubit passes through the channel untouched
            qubits_received_by_bob.append(qubits_sent[i])

    print(f"  Qubits intercepted : {n_intercepted} / {n_bits}")
    print(f"  Eve's bases (first 10): {eve_bases[:10]}")

    # ----------------------------------------------------------
    # BOB: choose random measurement bases, measure each qubit
    # (Bob receives either Alice's original qubit or Eve's forgery)
    # ----------------------------------------------------------
    print(f"\n[BOB]   Choosing {n_bits} random bases and measuring ...")
    bob_bases   = quantum_random_bits(n_bits)
    bob_results = [bob_measure(qc, basis)
                   for qc, basis in zip(qubits_received_by_bob, bob_bases)]

    print(f"  bases   (first 10): {bob_bases[:10]}")
    print(f"  results (first 10): {bob_results[:10]}")

    # ----------------------------------------------------------
    # PUBLIC CHANNEL: basis reconciliation
    # ----------------------------------------------------------
    print(f"\n[PUBLIC] Alice and Bob compare bases ...")
    matching_idx = [i for i in range(n_bits)
                    if alice_bases[i] == bob_bases[i]]
    print(f"  Matching positions: {len(matching_idx)} / {n_bits}")

    # ----------------------------------------------------------
    # SIFT: extract shared key from matching-basis positions
    # ----------------------------------------------------------
    alice_sifted = [alice_bits[i]   for i in matching_idx]
    bob_sifted   = [bob_results[i]  for i in matching_idx]
    sifted_len   = len(alice_sifted)
    print(f"\n[SIFT]  Sifted key length: {sifted_len} bits")

    # ----------------------------------------------------------
    # CHECK: sacrifice first ~20% of sifted key to estimate QBER
    # Eve's interceptions introduce errors here (~25% expected).
    # ----------------------------------------------------------
    check_n = max(1, sifted_len // 5)
    errors  = sum(a != b for a, b in
                  zip(alice_sifted[:check_n], bob_sifted[:check_n]))
    qber    = errors / check_n

    final_alice = alice_sifted[check_n:]
    final_bob   = bob_sifted[check_n:]
    key_len     = len(final_alice)

    print(f"\n[CHECK] Sacrificing {check_n} bits to estimate error rate ...")
    print(f"  Errors : {errors} / {check_n}")
    print(f"  QBER   : {qber:.2%}  (expected ~{n_intercepted/n_bits*25:.1f}% from Eve)")
    print(f"  Threshold: {error_threshold:.0%}")

    # ----------------------------------------------------------
    # REPORT: declare outcome
    # ----------------------------------------------------------
    attack_detected = qber > error_threshold
    print("\n[RESULT]")
    if attack_detected:
        print(f"  WARNING: ATTACK DETECTED (QBER {qber:.2%} > {error_threshold:.0%})")
        print("  Alice and Bob abort the key exchange.")
    else:
        print(f"  NOTE: Attack not detected (QBER {qber:.2%} <= {error_threshold:.0%})")
        print(f"  Key length: {key_len} bits  <- potentially compromised!")

    return {
        "qber"            : qber,
        "sifted_len"      : sifted_len,
        "key_len"         : key_len,
        "n_intercepted"   : n_intercepted,
        "attack_detected" : attack_detected,
    }


# ─────────────────────────────────────────────────────────────
#  MAIN
# ─────────────────────────────────────────────────────────────
N         = 100    # number of qubits exchanged
THRESHOLD = 0.15   # QBER threshold for attack detection (15%)

result = run_bb84_with_attacker(n_bits=N, error_threshold=THRESHOLD)

print("\n" + "=" * 58)
print("  SUMMARY")
print("=" * 58)
print(f"  Qubits exchanged : {N}")
print(f"  Eve intercepted  : {result['n_intercepted']} / {N}")
print(f"  Sifted key length: {result['sifted_len']} bits")
print(f"  Final key length : {result['key_len']} bits")
print(f"  QBER             : {result['qber']:.2%}")
print(f"  Attack detected  : {result['attack_detected']}")
print("=" * 58)

  BB84 — WITH EVE (intercept-resend attack)

[ALICE] Generating 100 random bits and bases ...
  bits  (first 10): [1, 1, 0, 1, 0, 1, 0, 0, 1, 1]
  bases (first 10): [0, 0, 1, 0, 1, 1, 1, 1, 1, 1]   (0=+, 1=x)

[EVE]   Intercepting qubits (quantum coin decides per qubit) ...
  Qubits intercepted : 56 / 100
  Eve's bases (first 10): [1, 1, 1, 0, 0, 0, 0, 1, 0, 1]

[BOB]   Choosing 100 random bases and measuring ...
  bases   (first 10): [1, 1, 0, 0, 0, 1, 0, 0, 0, 1]
  results (first 10): [0, 0, 1, 1, 1, 0, 0, 0, 0, 1]

[PUBLIC] Alice and Bob compare bases ...
  Matching positions: 48 / 100

[SIFT]  Sifted key length: 48 bits

[CHECK] Sacrificing 9 bits to estimate error rate ...
  Errors : 2 / 9
  QBER   : 22.22%  (expected ~14.0% from Eve)
  Threshold: 15%

[RESULT]
  Alice and Bob abort the key exchange.

  SUMMARY
  Qubits exchanged : 100
  Eve intercepted  : 56 / 100
  Sifted key length: 48 bits
  Final key length : 39 bits
  QBER             : 22.22%
  Attack detected  : True
